In [1]:
import cornac
from cornac.data import Reader
from cornac.datasets import movielens
from cornac.data import Dataset, FeatureModality
from cornac.eval_methods import RatioSplit, StratifiedSplit
from cornac.metrics import RMSE
from cornac.models import MF, ItemKNN, UserKNN, NMF, BPR
import pandas as pd
import numpy as np

import math
import seaborn as sns
import matplotlib.pyplot as plt


/Users/tahsinalamgirkheya/anaconda3/envs/cornac/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
reader = Reader()
rating_data = pd.read_csv(
    "./cornac/data_c/indexed_interactions.csv",
    sep="\t",
    header=None,
    names=["userID", "itemID", "Rating", "Timestamp"],
)
print(rating_data['itemID'].nunique())
# rating_data = rating_data.drop(columns=rating_data.columns[-1])
rating_data = rating_data.to_numpy()
rating_data
# user id itemid rating and timestamp

3416


array([[        0,         0,         5, 978300760],
       [        0,         1,         3, 978302109],
       [        0,         2,         3, 978301968],
       ...,
       [     6039,       365,         5, 956704746],
       [     6039,       152,         4, 956715648],
       [     6039,        26,         4, 956715569]])

In [3]:
# dataset = Dataset.from_uirt(rating_data, reader=reader)
dataset=rating_data


In [4]:
rating_data.shape

(999611, 4)

In [5]:
# movie_data = reader.read(fpath="./data/indexed_movies.csv", sep=",", fmt="UIRT")
# movie_data
movies = pd.read_csv("./cornac/data_c/indexed_movies.csv")

movies = movies.drop(columns=movies.columns[0])
movies[:4]

unique_genres = set("|".join(movies["genres"]).split("|"))
unique_genres = list(unique_genres)

for genre in unique_genres:
    movies[genre] = 0
for index, row in movies.iterrows():
    genres = row["genres"].split("|")
    for genre in genres:
        movies.at[index, genre] = 1

# item_categories = movies[unique_genres]
genre = movies[unique_genres]
item_features_numpy = genre.to_numpy()
# print(item_features_numpy.shape)

# item_categories = item_categor
item_features = {
    str(item_id): {"genre_" + str(idx): value for idx, value in enumerate(row)}
    for item_id, row in enumerate(item_features_numpy)
}
ids = list(range(0, 3416))
item_feature_modality = FeatureModality(
    features=item_features_numpy, ids=ids, normalized=True
)
item_feature_modality.__getstate__()
users = pd.read_csv("./cornac/data_c/u_id_mapping.csv", sep="\t")
users = users.drop(columns=users.columns[0])
gender_map = {"M": 0, "F": 1}
users["Gender"] = users["Gender"].map(gender_map)

user_features_numpy = users.to_numpy()
print(user_features_numpy.shape)
user_feature_modality = FeatureModality(
    features=user_features_numpy, name="user", normalized=True, ids=list(range(0, 6040))
)

# print("Example Item Features:")
# for item_id, features in list(item_features.items())[:5]:
#     print(f"Item ID: {item_id}, Features: {features}")

(6040, 2)


In [6]:
item_features_numpy[4]

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0])

In [7]:
# dataset.add_modalities(user_feature=user_feature_modality, item_features=item_feature_modality )
user_features_numpy

array([[   0,    1],
       [   1,    0],
       [   2,    0],
       ...,
       [6037,    1],
       [6038,    1],
       [6039,    0]])

In [8]:
ratio_split = StratifiedSplit(
    data=dataset, test_size=0.2, rating_threshold=0.0, seed=123, verbose=True, chrono=True,user_features = user_features_numpy[:,1]
)

model = MF(
    k=10, max_iter=50, learning_rate=0.001, lambda_reg=0.02, seed=123, name="lmd0.02", backend="pytorch", item_features=item_features_numpy
)

cornac.Experiment(
    ratio_split, models=[model], metrics=[cornac.metrics.RMSE(),cornac.metrics.AUC()]
).run()

rating_threshold = 0.0
exclude_unknowns = True
---
Training data:
Number of users = 6040
Number of items = 3415
Number of ratings = 797275
Max rating = 5.0
Min rating = 1.0
Global mean = 3.6
---
Test data:
Number of users = 6040
Number of items = 3415
Number of ratings = 202331
Number of unknown users = 0
Number of unknown items = 0
---
Total users = 6040
Total items = 3415

[lmd0.02] Training started!
-------------------
OrderedDict([(6039, 0), (6038, 1), (6037, 2), (6036, 3), (6035, 4), (6034, 5), (6033, 6), (6032, 7), (6031, 8), (6030, 9), (6029, 10), (6028, 11), (6027, 12), (6026, 13), (6025, 14), (6024, 15), (6023, 16), (6022, 17), (6021, 18), (6020, 19), (6019, 20), (6018, 21), (6017, 22), (6016, 23), (6015, 24), (6014, 25), (6013, 26), (6012, 27), (6011, 28), (6010, 29), (6009, 30), (6008, 31), (6006, 32), (6007, 33), (6005, 34), (6004, 35), (6003, 36), (6002, 37), (6001, 38), (6000, 39), (5999, 40), (5998, 41), (5997, 42), (5996, 43), (5995, 44), (5994, 45), (5993, 46), (5992, 

  0%|          | 0/50 [00:00<?, ?it/s, loss=1.21]

./././././././.
tensor([2., 4., 4., 4., 5., 4., 4., 4., 1., 4., 1., 3., 5., 4., 5., 3., 3., 3.,
        3., 5., 5., 2., 2., 4., 4., 4., 3., 4., 4., 5., 4., 1., 3., 3., 5., 5.,
        3., 3., 3., 2., 4., 3., 4., 4., 4., 2., 3., 4., 4., 3., 3., 4., 1., 4.,
        5., 4., 4., 4., 5., 5., 4., 3., 3., 3., 3., 4., 4., 3., 5., 5., 3., 4.,
        2., 4., 4., 1., 2., 5., 4., 4., 4., 5., 5., 3., 4., 3., 4., 3., 4., 4.,
        5., 4., 4., 4., 1., 2., 3., 4., 5., 3., 4., 2., 4., 4., 2., 5., 4., 5.,
        1., 4., 5., 3., 5., 4., 3., 4., 2., 2., 5., 1., 5., 1., 5., 3., 4., 4.,
        4., 3., 4., 5., 4., 5., 5., 3., 4., 4., 2., 4., 5., 3., 1., 4., 5., 2.,
        2., 3., 4., 1., 5., 4., 3., 4., 5., 4., 5., 3., 4., 3., 5., 5., 4., 5.,
        2., 2., 3., 3., 4., 4., 2., 4., 4., 1., 5., 4., 4., 5., 4., 4., 5., 4.,
        5., 3., 4., 3., 4., 2., 5., 4., 1., 5., 3., 4., 4., 5., 3., 4., 3., 1.,
        4., 3., 5., 5., 4., 4., 5., 4., 3., 3., 4., 1., 4., 4., 4., 3., 4., 3.,
        4., 4., 2., 5., 

  0%|          | 0/50 [00:00<?, ?it/s, loss=1.22]

tensor([4., 2., 5., 3., 2., 3., 2., 4., 4., 5., 4., 4., 5., 4., 5., 5., 2., 4.,
        3., 4., 1., 5., 1., 4., 5., 4., 3., 2., 4., 5., 4., 4., 5., 4., 4., 1.,
        4., 4., 2., 4., 2., 3., 5., 1., 4., 2., 3., 5., 5., 5., 4., 1., 4., 5.,
        5., 4., 2., 5., 2., 5., 5., 5., 5., 2., 5., 1., 3., 2., 5., 2., 4., 3.,
        4., 5., 5., 3., 4., 5., 2., 3., 4., 4., 5., 1., 5., 4., 2., 5., 1., 3.,
        3., 3., 5., 1., 4., 3., 2., 4., 4., 5., 3., 3., 5., 1., 4., 3., 5., 3.,
        3., 2., 5., 3., 4., 3., 3., 4., 3., 4., 4., 3., 3., 5., 5., 5., 5., 4.,
        5., 3., 5., 4., 3., 4., 4., 5., 4., 5., 5., 3., 5., 2., 3., 4., 4., 4.,
        4., 4., 5., 5., 5., 5., 1., 3., 5., 2., 4., 5., 1., 2., 2., 5., 1., 5.,
        3., 5., 4., 2., 4., 4., 1., 5., 4., 5., 4., 3., 4., 4., 3., 5., 3., 5.,
        3., 3., 4., 4., 3., 3., 5., 5., 4., 3., 4., 2., 3., 3., 3., 3., 1., 3.,
        4., 5., 1., 4., 3., 4., 5., 1., 5., 2., 4., 5., 3., 4., 2., 5., 3., 4.,
        4., 4., 5., 4., 1., 2., 3., 3., 

...

./././././././.
tensor([5., 3., 2., 4., 5., 2., 4., 2., 3., 2., 1., 3., 3., 5., 5., 4., 5., 4.,
        3., 3., 4., 1., 4., 2., 4., 4., 4., 5., 3., 4., 4., 4., 4., 5., 5., 4.,
        4., 5., 2., 4., 2., 5., 1., 3., 3., 3., 5., 3., 5., 2., 3., 3., 1., 4.,
        4., 3., 1., 4., 3., 4., 3., 5., 4., 5., 5., 4., 4., 4., 3., 5., 3., 4.,
        4., 3., 3., 5., 2., 4., 2., 3., 4., 5., 4., 4., 3., 4., 3., 4., 1., 5.,
        4., 3., 3., 5., 4., 3., 5., 1., 5., 4., 3., 4., 2., 2., 3., 4., 2., 4.,
        3., 5., 5., 4., 4., 3., 3., 4., 3., 5., 3., 2., 3., 3., 4., 3., 5., 4.,
        3., 4., 5., 4., 2., 2., 4., 1., 4., 4., 3., 4., 4., 4., 5., 4., 4., 5.,
        4., 4., 5., 3., 4., 3., 3., 5., 5., 2., 4., 4., 2., 3., 4., 5., 4., 4.,
        5., 3., 4., 4., 3., 3., 4., 3., 3., 5., 4., 4., 2., 5., 4., 3., 4., 4.,
        3., 5., 4., 5., 5., 3., 4., 4., 4., 4., 4., 5., 3., 1., 4., 1., 5., 3.,
        4., 4., 2., 4., 3., 4., 4., 4., 5., 5., 4., 2., 2., 1., 3., 4., 4., 5.,
        1., 4., 4., 3., 

  0%|          | 0/50 [00:01<?, ?it/s, loss=1.19]

./././././././.
tensor([5., 3., 5., 1., 3., 4., 4., 5., 4., 1., 3., 3., 3., 4., 5., 4., 4., 4.,
        4., 5., 3., 5., 4., 1., 1., 4., 2., 5., 4., 5., 3., 1., 5., 5., 3., 4.,
        4., 4., 5., 5., 5., 5., 4., 3., 4., 3., 5., 3., 3., 4., 5., 1., 1., 5.,
        5., 1., 5., 5., 4., 3., 5., 5., 3., 4., 1., 2., 5., 2., 4., 5., 5., 4.,
        4., 3., 5., 5., 5., 5., 4., 4., 4., 4., 3., 3., 5., 4., 5., 5., 4., 4.,
        5., 5., 5., 5., 4., 2., 3., 4., 2., 2., 5., 3., 5., 5., 4., 1., 5., 3.,
        4., 4., 3., 5., 4., 4., 4., 5., 4., 3., 4., 5., 2., 4., 4., 3., 3., 4.,
        5., 5., 4., 5., 1., 3., 4., 4., 2., 4., 4., 5., 4., 4., 5., 5., 3., 4.,
        4., 4., 4., 3., 2., 4., 3., 4., 4., 3., 4., 3., 4., 4., 2., 2., 4., 3.,
        2., 5., 3., 5., 2., 5., 2., 3., 5., 5., 4., 3., 5., 5., 3., 3., 4., 4.,
        4., 4., 3., 4., 3., 1., 3., 5., 3., 1., 5., 5., 5., 4., 2., 2., 5., 5.,
        3., 4., 5., 4., 4., 3., 2., 4., 4., 3., 3., 4., 5., 3., 5., 3., 4., 5.,
        2., 4., 4., 5., 

  0%|          | 0/50 [00:01<?, ?it/s, loss=1.19]


KeyboardInterrupt: 

In [ ]:
user_ids = users.to_numpy()[:, 0]
item_ids = movies.to_numpy()[:, 2]

In [ ]:
# get the top_k ratings for all users:
top_k=50
reco_matrix = np.zeros((len(user_ids), top_k), dtype=int)

for u in user_ids:
    reco_items, all_scores = model.rank(user_idx=u, item_indices=list(item_ids))
    reco_matrix[u] = reco_items[:top_k]

In [ ]:
####IR Metrics####
####Precision####
from cornac.mymetrics.GenrePrecision import GenrePrecision

gp = GenrePrecision(users, unique_genres)
MF_gp = gp.compute(reco_matrix, movies)

In [ ]:
rating_data

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

matrices = [MF_gp]


values1 = matrices[0]
labels = unique_genres
bar_width = 0.25
titles = ["MF_gp"]


fig, axs = plt.subplots(2, 3, figsize=(18, 12))
axs = axs.flatten()


for i in range(len(matrices)):
    axs[i].bar(labels, matrices[i], color="skyblue")
    axs[i].set_title(titles[i])
    axs[i].set_xlabel("Genres")
    axs[i].set_ylabel("Values")
    axs[i].tick_params(
        axis="x", rotation=90
    ) 

# Adjust layout
plt.tight_layout()
plt.show()

In [ ]:
item_features_numpy.shape

In [ ]:
dataset


In [ ]:
        |   RMSE |    AUC | Train (s) | Test (s)
------- + ------ + ------ + --------- + --------
lmd0.02 | 1.0251 | 0.6337 |  117.4954 |   3.6938

In [ ]:
print(type(user_features_numpy[:,1]))

In [ ]:
print(user_features_numpy[:,1].shape)